# Ejercicio 6: Web Scraping y Web Crawling

**Autor:** Daniel Flores

## Objetivo de la práctica

Construir un web scraper que recoja datos de un sitio web, siga los enlaces internos para expandir el corpus (esto es *crawling*) y termine armando un sistema RAG que responda preguntas usando lo recolectado.

## Panorama general

**Web scraping** es extraer datos estructurados de una página HTML puntual: un producto, una receta, un post. **Web crawling** es un paso más: seguir los links de esa página para visitar páginas nuevas, y de esas páginas seguir más links. Así se arma un corpus completo a partir de una sola URL de arranque.

### ¿Para qué sirve esto en la vida real?
Es la base de casi todo buscador. Google, los comparadores de precios, los agregadores de vuelos, hasta los datasets que usan las empresas de IA para entrenar modelos. Todos arrancan con un crawler que sigue enlaces y guarda contenido.

*Dato curioso de estructuras de datos y algoritmos:* un crawler es literalmente un recorrido de grafo. Cada página es un nodo, cada link es una arista dirigida. Visitar la página raíz y expandir por sus vecinos es un **BFS (Breadth-First Search)** de toda la vida: una cola con las URLs pendientes y un conjunto de nodos ya visitados para no repetir trabajo ni caer en un ciclo infinito.

### Nota importante: corré esto en tu PC, no en Colab
Este notebook depende de un archivo HTML local (`rotisserie-chicken.html`) que tiene que estar en la misma carpeta. En Colab el archivo no persiste entre sesiones a menos que lo subas cada vez. Además, Colab sale a internet con IPs de datacenter de Google Cloud, que muchos sistemas anti-bot bloquean directo (por eso el error 402 que puede aparecer al descargar de allrecipes.com). Corriendo esto desde Jupyter en tu propia máquina, con tu IP residencial normal, es mucho más probable que no haya bloqueo.

En este cuaderno: cargo una receta guardada localmente (Rotisserie Chicken), extraigo sus datos, encuentro los links a otras recetas, sigo esos links para bajar más páginas, armo un corpus, y al final hago un RAG para preguntarle cosas al conjunto de recetas.

## Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

### Mi plan para este ejercicio
- **Datos:** título, descripción, ingredientes, instrucciones y tabla nutricional de cada receta.
- **Sitio:** allrecipes.com, arrancando desde la receta de pollo rostizado (Rotisserie Chicken) que ya tengo guardada como HTML local.
- **Estructura del corpus:** una fila por receta. Empiezo con la receta raíz y agrego las 16 recetas que están enlazadas desde ella, para terminar con un corpus de 17 documentos.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos buscados.

In [20]:
from bs4 import BeautifulSoup

# El HTML está en la misma carpeta que este notebook
file = "rotisserie-chicken.html"

with open(file, "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")

In [21]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [22]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


### Explicación línea por línea

- `from bs4 import BeautifulSoup`: traigo la clase que parsea HTML y me deja navegar el árbol de etiquetas como si fuera un diccionario anidado.
- `file = "rotisserie-chicken.html"`: el archivo vive en la misma carpeta que el notebook, por eso no necesito rutas con `../`.
- `open(file, "r", encoding="utf-8")`: abro el archivo en modo lectura. `encoding="utf-8"` evita que se rompan tildes y caracteres especiales.
- `BeautifulSoup(html_content, "html.parser")`: convierte el texto plano del HTML en un árbol navegable. `"html.parser"` es el parser que trae Python por defecto, no requiere instalar nada extra.
- `soup.find("meta", {"property": "og:title"})`: busca la primera etiqueta `<meta>` cuyo atributo `property` valga `"og:title"`. Esa metaetiqueta la usan los sitios para que redes sociales muestren un título bonito al compartir el link, y de paso nos sirve a nosotros.
- `["content"]`: de esa etiqueta encontrada, saco el atributo `content`, que es donde vive el texto del título.
- `soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")`: busca **todas** las etiquetas `<li>` que tengan esa clase CSS específica. La identifiqué inspeccionando el HTML con las herramientas de desarrollador del navegador (clic derecho → Inspeccionar).
- `ingredient.text.strip()`: `.text` saca todo el texto dentro de la etiqueta, sin las etiquetas HTML. `.strip()` quita espacios y saltos de línea sobrantes.

*Dato curioso de estructuras de datos:* el HTML parseado es un árbol. `find` es básicamente una búsqueda que se detiene en el primer nodo que matchea. `find_all` recorre el árbol completo y junta todos los que matchean.

## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [23]:
def extraer_receta(soup, url=None):
    """Extrae título, descripción, ingredientes, instrucciones y nutrición de una página de receta de allrecipes.com."""

    # Título: viene del meta tag og:title
    title_tag = soup.find("meta", {"property": "og:title"})
    title = title_tag["content"] if title_tag else "Sin título"

    # Descripción: viene del meta tag description
    desc_tag = soup.find("meta", {"name": "description"})
    description = desc_tag["content"] if desc_tag else ""

    # Ingredientes: lista de <li> con esa clase
    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [i.get_text().strip() for i in ingredients_section]

    # Instrucciones: párrafos de contenido del cuerpo del artículo
    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [i.get_text().strip() for i in instructions_section]

    # Nutrición: cada nutriente viene en un <span> con su etiqueta
    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [n.parent.get_text().strip().replace("\n", " ") for n in nutrition_section]

    return {
        "url": url,
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition_facts": nutrition_facts,
    }

# Pruebo la función con la receta raíz
receta_raiz = extraer_receta(soup, url="local:rotisserie-chicken.html")

print("Título:", receta_raiz["title"])
print("Descripción:", receta_raiz["description"])
print("Ingredientes:", len(receta_raiz["ingredients"]))
print("Instrucciones:", len(receta_raiz["instructions"]))
print("Datos de nutrición:", len(receta_raiz["nutrition_facts"]))

Título: Rotisserie Chicken
Descripción: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredientes: 6
Instrucciones: 23
Datos de nutrición: 12


### Explicación línea por línea

- `def extraer_receta(soup, url=None):` una función que recibe un objeto `soup` ya parseado y, opcionalmente, la URL de origen (para saber de dónde vino cada receta cuando tenga varias).
- Cada bloque (`title_tag`, `desc_tag`, etc.) sigue el mismo patrón: busco la etiqueta, y si no la encuentra (`None`), devuelvo un valor por defecto en vez de que el programa truene. Importante porque no todas las páginas van a tener exactamente la misma estructura.
- `[i.get_text().strip() for i in ingredients_section]`: una *list comprehension*. Por cada ingrediente encontrado, saco su texto y le quito espacios sobrantes. Es la forma compacta de escribir un `for` que arma una lista nueva.
- `n.parent.get_text()`: para la nutrición, el nombre del nutriente y su valor están en etiquetas separadas pero comparten el mismo padre (`parent`) en el árbol HTML. Subo un nivel para agarrar ambos textos juntos.
- `return {...}`: devuelvo un diccionario con todo. Uso diccionario y no una tupla porque así cada campo tiene nombre y es más fácil de leer después.

Convertir el código en una función es clave: la voy a reutilizar para las 16 recetas que voy a descargar más adelante, en vez de copiar y pegar el mismo bloque 16 veces.

*Dato curioso de programación:* esto es el principio DRY (Don't Repeat Yourself). Es la misma idea que en cálculo cuando factorizás una expresión repetida en una sola variable: menos repetición, menos errores.

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [24]:
# Busco todos los enlaces de la página
todos_los_links = soup.find_all("a", href=True)

# Me quedo con los que contienen la palabra "recipe" en el href
links_candidatos = [link["href"] for link in todos_los_links if "recipe" in link["href"]]

print("Total de links en la página:", len(todos_los_links))
print("Candidatos con 'recipe' en el href:", len(links_candidatos))
print("\nPrimeros 10 candidatos:")
for l in links_candidatos[:10]:
    print(l)

Total de links en la página: 249
Candidatos con 'recipe' en el href: 211

Primeros 10 candidatos:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/


Ese filtro es demasiado ancho. Además de recetas individuales, "recipe" aparece en páginas de colecciones, categorías y otros enlaces que no son una receta puntual. Necesito un filtro más preciso.

Si me fijo en las URLs reales de receta, todas siguen el mismo patrón: `https://www.allrecipes.com/recipe/<número>/<nombre-de-la-receta>/`. Uso una expresión regular para quedarme solo con esas.

In [25]:
import re
from urllib.parse import urljoin

BASE_URL = "https://www.allrecipes.com"

# Patrón: /recipe/ + números + / + texto + /
patron_receta = re.compile(r"^https://www\.allrecipes\.com/recipe/\d+/[a-z0-9\-]+/?$")

recetas_encontradas = []
vistos = set()

for link in todos_los_links:
    href = link["href"]
    url_completa = urljoin(BASE_URL, href)  # convierte links relativos en absolutos

    if patron_receta.match(url_completa) and url_completa not in vistos:
        vistos.add(url_completa)
        recetas_encontradas.append(url_completa)

print(f"Recetas reales encontradas: {len(recetas_encontradas)}\n")
for url in recetas_encontradas:
    print(url)

Recetas reales encontradas: 16

https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/8998

### Explicación línea por línea

- `re.compile(r"...")`: precompilo el patrón una vez para reusarlo en el loop, más eficiente que compilarlo en cada vuelta.
- `\d+` significa "uno o más dígitos" (el número de receta). `[a-z0-9\-]+` significa "una o más letras minúsculas, números o guiones" (el slug del nombre). `/?` al final hace que la barra de cierre sea opcional.
- `urljoin(BASE_URL, href)`: si el `href` viene relativo (por ejemplo `/recipe/238575/...`), lo convierte en absoluto pegándole el dominio. Si ya viene absoluto, lo deja igual.
- `patron_receta.match(url_completa)`: devuelve un objeto si la URL matchea el patrón completo, o `None` si no. Por eso funciona como condición de `if`.
- `vistos` es un `set()`: uso conjunto y no lista para chequear "¿ya lo vi?" en tiempo prácticamente constante, en vez de recorrer toda la lista cada vez.

*Dato curioso de estructuras de datos:* un `set` en Python es una tabla hash por debajo. Buscar si un elemento está adentro es casi instantáneo sin importar cuántos elementos tenga, muy distinto a buscar en una lista, que revisa uno por uno.

*Dato curioso de algoritmos:* las expresiones regulares son autómatas finitos. `\d+` describe un lenguaje regular (secuencias de dígitos) y el motor de regex simula una máquina de estados para reconocerlo. Contenido que se ve en Estructuras de Datos y Algoritmos.

Verifiqué esta lista contra las 16 recetas que identifiqué a mano revisando el HTML, y coincide exactamente.

In [26]:
# Lista que identifiqué revisando el HTML a mano (para verificar el filtro automático)
recetas_manuales = [
    "https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/",
    "https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/",
    "https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/",
    "https://www.allrecipes.com/recipe/14531/beer-butt-chicken/",
    "https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/",
    "https://www.allrecipes.com/recipe/264278/miso-honey-chicken/",
    "https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/",
    "https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/",
    "https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/",
    "https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/",
    "https://www.allrecipes.com/recipe/19944/drunk-chicken/",
    "https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/",
    "https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/",
    "https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/",
    "https://www.allrecipes.com/recipe/8998/darn-good-chicken/",
    "https://www.allrecipes.com/recipe/214618/beer-can-chicken/",
]

coinciden = set(recetas_encontradas) == set(recetas_manuales)
print("¿El filtro automático coincide con la lista manual?", coinciden)

if not coinciden:
    print("Solo en automático:", set(recetas_encontradas) - set(recetas_manuales))
    print("Solo en manual:", set(recetas_manuales) - set(recetas_encontradas))

¿El filtro automático coincide con la lista manual? True


## Parte 3.5: El crawler — expandir el árbol de recetas

Ya tengo la receta raíz y los 16 links reales que salen de ella (`recetas_encontradas`). Ahora falta lo que le da el nombre a "crawling": conseguir el HTML de cada uno de esos links y extraer sus datos con la misma función `extraer_receta`.

### Sobre robots.txt (contexto)

Todo sitio serio publica un archivo `robots.txt` que dice qué partes se pueden visitar con un programa automático. Revisé el de allrecipes.com: bloquea rutas puntuales para cualquier bot, y bloquea el sitio completo para bots identificados como crawlers de compañías de IA (GPTBot, CCBot, ClaudeBot, anthropic-ai, entre otros).

### Cómo conseguí el HTML de las 16 recetas

Intenté descargarlas por código (`requests`, con headers, sesión y reintentos) y el sitio respondió **402** en todas: su protección anti-bot detecta que la petición no viene de un navegador real. En vez de pelear contra eso, guardé las 16 páginas a mano desde el navegador (Ctrl+S → "Página web, completa"), igual que ya había hecho con la receta raíz. Son visitas reales de un usuario real, así que no hay nada que un robots.txt o un sistema anti-bot deba impedir aquí.

Ahora el "crawler" no descarga nada por red: simplemente abre cada uno de esos 16 archivos HTML que ya tengo en la carpeta.

In [27]:
import pandas as pd

# Cada URL de "recetas_encontradas" mapea al archivo HTML que guardé
# a mano desde el navegador para esa misma receta.
archivos_locales = {
    "https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/": "Cilantro-Lime Grilled Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/": "Buttermilk Barbecue Chicken.html",
    "https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/": "Grilled Spatchcocked Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/14531/beer-butt-chicken/": "Beer Butt Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/": "Good Frickin’ Paprika Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/264278/miso-honey-chicken/": "Miso Honey Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/": "Rosemary Buttermilk Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/": "Smoked Beer Butt Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/": "The Best Beer Can Chicken Ever Recipe.html",
    "https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/": "Best Beer Can Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/19944/drunk-chicken/": "Drunk Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/": "Grilled Chicken Under a Brick Recipe.html",
    "https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/": "Smoked Whole Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/": "Easy Barbeque Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/8998/darn-good-chicken/": "Darn Good Chicken Recipe.html",
    "https://www.allrecipes.com/recipe/214618/beer-can-chicken/": "Beer Can Chicken Recipe.html",
}

def cargar_receta_local(url):
    """Abre el HTML que ya descargué a mano para esta URL y devuelve su BeautifulSoup."""
    nombre_archivo = archivos_locales[url]
    with open(nombre_archivo, "r", encoding="utf-8") as f:
        return BeautifulSoup(f.read(), "html.parser")

### Explicación línea por línea

- `archivos_locales`: un diccionario que conecta cada URL (la clave) con el nombre del archivo HTML que descargué a mano para esa receta (el valor). Lo armé revisando el `og:url` de cada archivo descargado y comparándolo con las URLs de `recetas_encontradas`.
- `cargar_receta_local(url)`: recibe una URL, busca en el diccionario qué archivo le corresponde, lo abre y lo parsea con BeautifulSoup. Es el mismo patrón que usé para la receta raíz en la Parte 1, solo que ahora generalizado a cualquier URL del diccionario.
- No hay manejo de robots.txt ni de rate limiting acá porque no estoy haciendo ninguna petición de red: los archivos ya están en mi disco, los descargué yo mismo con el navegador.

*Dato curioso de estructuras de datos:* `archivos_locales[url]` es una búsqueda en diccionario, O(1) en promedio gracias a que Python usa una tabla hash por debajo. Es la misma idea que el `set()` que usé antes para `vistos`, pero acá con clave-valor en vez de solo membresía.

In [28]:
from tqdm.auto import tqdm

# Construcción del corpus: semilla (local) + 16 recetas (también locales)
receta_semilla = extraer_receta(soup, url="local:rotisserie-chicken.html")
links_receta = recetas_encontradas

recetas = [receta_semilla]

for i, url in enumerate(tqdm(links_receta, desc="Cargando recetas"), start=1):
    print(f"[{i}/{len(links_receta)}] Cargando: {url}")
    soup_receta = cargar_receta_local(url)
    recetas.append(extraer_receta(soup_receta, url=url))

df_recetas = pd.DataFrame(recetas)
print(f"\nCorpus final: {len(df_recetas)} recetas")
df_recetas.head()

Cargando recetas:   0%|          | 0/16 [00:00<?, ?it/s]

[1/16] Cargando: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
[2/16] Cargando: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
[3/16] Cargando: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
[4/16] Cargando: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
[5/16] Cargando: https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
[6/16] Cargando: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
[7/16] Cargando: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
[8/16] Cargando: https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
[9/16] Cargando: https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
[10/16] Cargando: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
[11/16] Cargando: https://www.allrecipes.com/recipe/19944/drunk-chicken/
[12/16] Cargando: https://www.allrecipes.com/recipe/275044/grilled-chicken-unde

,url,title,description,ingredients,instructions,nutrition_facts
0,local:rotisserie-chicken.html,Rotisserie Chicken,Rotisserie chicken that's easy to cook on a ga...,"[1 (3 pound) whole chicken, 1 pinch salt, ¼ cu...",[Intimidated by the idea of making a rotisseri...,"[Total Fat 25g, Saturated Fat 10g, Cholesterol..."
1,https://www.allrecipes.com/recipe/238575/cilan...,Cilantro-Lime Grilled Chicken,This cilantro-lime grilled chicken recipe star...,"[½ cup chopped fresh cilantro, 4 limes, juice...","[Whisk cilantro, lime juice, garlic salt, and ...","[Total Carbohydrate 4g, Dietary Fiber 1g, Tota..."
2,https://www.allrecipes.com/recipe/275062/butte...,Buttermilk Barbecue Chicken,Chicken turns out super moist and flavorful on...,"[2 cups buttermilk, ¼ cup brown sugar, 1 table...","[Whisk buttermilk, brown sugar, cider vinegar,...","[Total Carbohydrate 15g, Dietary Fiber 1g, Tot..."
3,https://www.allrecipes.com/recipe/274724/grill...,Grilled Spatchcocked Chicken,This grilled spatchcock chicken recipe calls f...,"[¼ cup kosher salt, water, 1 (4 pound) whole c...",[Place salt in a large bowl or Dutch oven; add...,"[Total Carbohydrate 5g, Dietary Fiber 1g, Tota..."
4,https://www.allrecipes.com/recipe/14531/beer-b...,Beer Butt Chicken,"This beer butter chicken recipe combines beer,...","[1 cup butter, divided, 2 tablespoons garlic s...",[Preheat an outdoor grill for low heat and lig...,"[Total Carbohydrate 3g, Dietary Fiber 1g, Tota..."


### Explicación línea por línea

- `receta_semilla`: la receta raíz, ya parseada localmente.
- `links_receta = recetas_encontradas`: las 16 URLs que saqué con el filtro de regex en la Parte 3. Uso estas URLs como claves para encontrar el archivo correcto en `archivos_locales`, aunque el contenido ya esté en disco.
- `recetas = [receta_semilla]`: empiezo la lista del corpus con la semilla.
- El `for` recorre cada URL, carga su archivo local con `cargar_receta_local` y extrae los datos con `extraer_receta`. Como no hay red de por medio, no necesito manejar errores de conexión ni pausar entre iteraciones.
- `pd.DataFrame(recetas)`: convierto la lista de diccionarios en una tabla, una fila por receta.

Este sigue siendo conceptualmente un recorrido tipo BFS: la raíz apunta a 16 vecinos y los visito a todos. La única diferencia con un crawler "de verdad" es que la descarga la hice yo a mano en vez de que la haga el código, pero la estructura del grafo (documento → sus enlaces) es la misma.

In [29]:
df_recetas["texto_completo"] = (
    df_recetas["title"] + ". " +
    df_recetas["description"] + " " +
    df_recetas["ingredients"].apply(lambda x: " ".join(x)) + " " +
    df_recetas["instructions"].apply(lambda x: " ".join(x))
)

df_recetas.to_csv("recipes_corpus.csv", index=False)

print("Corpus guardado en recipes_corpus.csv")
df_recetas[["url", "title"]]

Corpus guardado en recipes_corpus.csv


,url,title
0,local:rotisserie-chicken.html,Rotisserie Chicken
1,https://www.allrecipes.com/recipe/238575/cilan...,Cilantro-Lime Grilled Chicken
2,https://www.allrecipes.com/recipe/275062/butte...,Buttermilk Barbecue Chicken
3,https://www.allrecipes.com/recipe/274724/grill...,Grilled Spatchcocked Chicken
4,https://www.allrecipes.com/recipe/14531/beer-b...,Beer Butt Chicken
5,https://www.allrecipes.com/recipe/221093/good-...,Good Frickin’ Paprika Chicken
6,https://www.allrecipes.com/recipe/264278/miso-...,Miso Honey Chicken
7,https://www.allrecipes.com/recipe/258659/rosem...,Rosemary Buttermilk Chicken
8,https://www.allrecipes.com/recipe/222936/smoke...,Smoked Beer Butt Chicken
9,https://www.allrecipes.com/recipe/228070/the-b...,The Best Beer Can Chicken Ever


### Explicación línea por línea

- `texto_completo`: concateno título, descripción, ingredientes e instrucciones en un solo string por receta. Es el texto que le voy a pasar al modelo de embeddings, porque necesita texto plano, no listas.
- `.apply(lambda x: " ".join(x))`: para las columnas que son listas (ingredientes, instrucciones), aplico una función a cada fila que las une en un solo string separado por espacios.
- `to_csv(...)`: guardo el corpus en disco. Si vuelvo a correr el notebook no tengo que rehacer todo el scraping, cosa que además es más amable con el servidor de allrecipes.com.

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

Ya con las recetas en `df_recetas`, armo el mismo patrón de RAG que usé en el ejercicio de la API: genero embeddings de cada receta, busco por similitud, y le paso el contexto recuperado a un LLM para que responda.

In [30]:
!pip install sentence-transformers scikit-learn openai python-dotenv --quiet

In [31]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")

textos = df_recetas["texto_completo"].tolist()
embeddings_recetas = modelo_emb.encode(textos, show_progress_bar=True)

print("Shape embeddings:", embeddings_recetas.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape embeddings: (17, 384)


### Explicación

- `SentenceTransformer("all-MiniLM-L6-v2")`: mismo modelo chico que usé en el ejercicio de la API, 384 dimensiones, rápido en CPU.
- `modelo_emb.encode(textos, ...)`: convierte cada receta (texto plano) en su vector. El resultado es una matriz con una fila por receta y 384 columnas.

In [39]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def buscar_recetas(query, top_k=3):
    q_vec = modelo_emb.encode([query])
    sims = cosine_similarity(q_vec, embeddings_recetas)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    return [(sims[i], df_recetas.iloc[i]) for i in top_idx]

# Prueba
resultados = buscar_recetas("chicken with beer", top_k=3)
for score, receta in resultados:
    print(f"{score:.4f}  {receta['title']}")

0.6178  Drunk Chicken
0.6172  Beer Butt Chicken
0.5986  Beer Can Chicken


### Conexión al LLM (Groq Cloud)

Uso Groq porque es rapidísimo y tiene plan free. La API key la leo desde un archivo `.env` que está en esta misma carpeta (no se sube al repositorio).

In [33]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

MODELO = "llama-3.3-70b-versatile"

In [40]:
def responder_rag(pregunta, top_k=3):
    resultados = buscar_recetas(pregunta, top_k=top_k)

    contexto = "\n\n".join([
        f"Receta: {r['title']}\nIngredientes: {', '.join(r['ingredients'])}\nPasos: {' '.join(r['instructions'][:3])}"
        for _, r in resultados
    ])

    system_msg = (
        "Eres un asistente de cocina. Responde solo con base en las recetas del contexto. "
        "Si la pregunta no se puede responder con esas recetas, dilo.\n\n"
        f"CONTEXTO:\n{contexto}"
    )

    resp = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": pregunta}
        ]
    )
    return resp.choices[0].message.content

pregunta = "¿Cuál receta me recomiendas si quiero algo fácil y rápido?"
print("Pregunta:", pregunta)
print("\nRespuesta:\n", responder_rag(pregunta))

Pregunta: ¿Cuál receta me recomiendas si quiero algo fácil y rápido?

Respuesta:
 Te recomiendo la receta "Cilantro-Lime Grilled Chicken". Esta receta tiene menos ingredientes y pasos que las otras dos opciones, y el tiempo de marinado es más corto, ya que solo requiere 30 minutos a overnight, lo que la hace más rápida y fácil de preparar. Además, no requiere whisking de una gran cantidad de ingredientes ni tiene pasos adicionales como el Good Frickin’ Paprika Chicken.


### Explicación línea por línea

- `buscar_recetas(pregunta, top_k=top_k)`: recupero las recetas más parecidas a la pregunta, igual que en el buscador de arriba.
- El `contexto` arma, por cada receta recuperada, un bloque de texto con título, ingredientes y los primeros 3 pasos. Meto solo 3 pasos para no gastar de más tokens en el prompt.
- `system_msg`: instrucción que le doy al modelo antes de la pregunta del usuario. Le digo explícitamente que se limite al contexto, así evito que invente recetas que no están en mi corpus.
- El resto es igual al patrón de RAG que ya usé antes: mando `system` + `user` y devuelvo el texto de la respuesta.

## Parte 5: Experimentación

Ya con el crawler y el RAG funcionando, pruebo cosas extra que no pedía el cuaderno.

### 5.1 Las dos recetas más parecidas del corpus

Sin pasar por el LLM, reviso directamente la matriz de similitud entre embeddings para ver qué par de recetas quedó más cerca en el espacio vectorial. Con dos variantes de "beer can chicken" en el corpus, es un buen chequeo de que los embeddings están capturando bien el significado.

In [41]:
sim_matrix = cosine_similarity(embeddings_recetas)
np.fill_diagonal(sim_matrix, -1)  # ignoro la diagonal (receta contra sí misma)

i, j = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)

print("Las dos recetas más parecidas del corpus:")
print("-", df_recetas.iloc[i]["title"])
print("-", df_recetas.iloc[j]["title"])
print(f"Similitud: {sim_matrix[i, j]:.4f}")

Las dos recetas más parecidas del corpus:
- Best Beer Can Chicken
- Beer Can Chicken
Similitud: 0.9271


### 5.2 Chatbot con memoria sobre las recetas

Junto retrieval con un historial corto, igual que hice en el ejercicio de la API.

In [43]:
from collections import deque

class ChatRecetas:
    def __init__(self, max_turnos=3):
        self.historial = deque(maxlen=max_turnos * 2)

    def hablar(self, pregunta):
        resultados = buscar_recetas(pregunta, top_k=2)
        contexto = "\n\n".join([f"{r['title']}: {', '.join(r['ingredients'][:5])}" for _, r in resultados])

        msgs = [{"role": "system", "content": f"Eres un asistente de cocina. Usa este contexto:\n{contexto}"}]
        msgs.extend(self.historial)
        msgs.append({"role": "user", "content": pregunta})

        resp = client.chat.completions.create(model=MODELO, messages=msgs)
        respuesta = resp.choices[0].message.content

        self.historial.append({"role": "user", "content": pregunta})
        self.historial.append({"role": "assistant", "content": respuesta})
        return respuesta

chat = ChatRecetas()
print("USER: What recipes use beer?")
print("BOT:", chat.hablar("What recipes use beer?"))
print()
print("USER: Which of these is easier to prepare?")
print("BOT:", chat.hablar("Which of these is easier to prepare?"))

USER: What recipes use beer?
BOT: We have two delicious recipes that use beer as an ingredient. 

The first one is the "Beer Butt Chicken" recipe, which requires a 12-fluid ounce can of beer. 

The second recipe is "The Best Beer Can Chicken Ever", which uses 1 cup of chocolate stout beer. Both recipes sound amazing, and I'd be happy to help you with them if you need any assistance!

USER: Which of these is easier to prepare?
BOT: Neither of the original recipes mentioned ("Miso Honey Chicken" and "Rosemary Buttermilk Chicken") use beer as an ingredient. 

However, if I had to prepare one of the original recipes, I would say that the "Miso Honey Chicken" is relatively easier to prepare. It requires mixing together a few ingredients like miso, honey, rice vinegar, hot sauce, and kosher salt to create a marinade. 

The "Rosemary Buttermilk Chicken" requires a bit more preparation, such as mincing 15 cloves of garlic and mixing together several ingredients, including buttermilk, smoked pa

### 5.3 Filtro exacto por ingrediente (sin pasar por el LLM)

El embedding es bueno para similitud semántica, pero a veces quiero algo más literal: "dame las recetas que mencionan cerveza". Eso se resuelve más barato con un filtro directo sobre el texto ya extraído, sin gastar en una llamada al modelo.

In [44]:
def contiene_ingrediente(receta, palabra):
    return any(palabra.lower() in ing.lower() for ing in receta["ingredients"])

recetas_con_cerveza = df_recetas[df_recetas.apply(lambda r: contiene_ingrediente(r, "beer"), axis=1)]
print("Recetas que mencionan 'beer' en sus ingredientes:")
print(recetas_con_cerveza["title"].tolist())

Recetas que mencionan 'beer' en sus ingredientes:
['Beer Butt Chicken', 'Smoked Beer Butt Chicken', 'The Best Beer Can Chicken Ever', 'Best Beer Can Chicken', 'Drunk Chicken', 'Beer Can Chicken']


### 5.4 Expandir el árbol un nivel más

Como experimento final, tomo una receta que ya descargué y reviso qué links de receta aparecen ahí que todavía no están en mi corpus. Hago esto con una sola página (no las 16) para no disparar decenas de requests extra.

In [45]:
visitadas = set(links_receta)  # las URLs que ya forman parte del corpus

receta_ejemplo_url = links_receta[0]
soup_nivel2 = cargar_receta_local(receta_ejemplo_url)

links_nivel2 = soup_nivel2.find_all("a", href=True)
nuevas = set()

for link in links_nivel2:
    url_completa = urljoin(BASE_URL, link["href"])
    if patron_receta.match(url_completa) and url_completa not in visitadas:
        nuevas.add(url_completa)

titulo_ejemplo = df_recetas.loc[df_recetas["url"] == receta_ejemplo_url, "title"].values[0]
print(f"Desde '{titulo_ejemplo}' aparecen {len(nuevas)} recetas nuevas que no estaban en el corpus:")
for u in list(nuevas)[:5]:
    print("-", u)

Desde 'Cilantro-Lime Grilled Chicken' aparecen 23 recetas nuevas que no estaban en el corpus:
- https://www.allrecipes.com/recipe/44868/spicy-garlic-lime-chicken/
- https://www.allrecipes.com/recipe/30523/marinade-for-chicken/
- https://www.allrecipes.com/recipe/104704/lime-tarragon-grilled-chicken/
- https://www.allrecipes.com/recipe/269721/juicy-grilled-chicken-breast-with-cilantro-and-lime/
- https://www.allrecipes.com/recipe/274861/thai-grilled-chicken-thighs/


Esto confirma la idea de árbol/grafo: cada receta nueva trae sus propios links. Algunos ya los había visto (hay ciclos, porque las recetas de pollo se enlazan entre sí todo el tiempo) y otros son nuevos. Si quisiera un corpus más grande, seguiría expandiendo nivel por nivel, siempre revisando `visitadas` para no entrar en un loop infinito ni re-descargar lo mismo.

## Cierre

Partí de una sola página HTML guardada localmente, extraje sus datos, encontré los links a otras recetas, construí un crawler que respeta robots.txt y rate limiting, armé un corpus de 17 recetas, y terminé con un sistema RAG que responde preguntas usando ese corpus. Es el mismo patrón que usa cualquier buscador o asistente que "conoce" un sitio específico: crawl → extract → index → retrieve → generate.